# Quantum Yield Resource Estimates

Implementation for estimating logical qubits, ancilla qubits, and Toffoli counts for the quantum-yield workflow.

Notation used consistently throughout this notebook:

- **Toffoli count**: number of Toffoli gates.
- **Qubit count**: number of logical qubits.
- **Ancilla qubits**: auxiliary logical qubits beyond the state-vector register.
- **Error budget**: allocated approximation error for a specific algorithmic component.


In [1]:
import math
import sys
from pathlib import Path
from datetime import datetime

import numpy as np
from scipy.integrate import quad

# Local module paths
current_dir = Path.cwd()
root_dir = current_dir.parent
src_dir = root_dir / "src"
lambda_dir = current_dir / "Lambda"

for path in (src_dir, lambda_dir):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.append(path_str)

import charge_ratio
import io_functions as io
import lambda_molecular as lam
import lookup_table as lam_nu
import parse_isp
import parse_psi4
import physical_consts

## 1. Error budget allocation

Allocate the total quantum-yield error budget across initial state preparation, QPE, QSP, and Hamiltonian-simulation subroutines.


In [2]:
def get_error_budget(
    error_init,
    error_ham,
    error_qft_data,
    error_qft_qpe,
    error_qsp,
    error_yield,
    sim_time,
    lambda_ham,
    weight_theta=1.0 / 3.0,
    weight_T=1.0 / 3.0,
    weight_V=1.0 / 3.0,
    error_rot=1e-22,
    error_gamma=1e-25,
    error_phi=1e-25
):
    """Allocate the quantum-yield error budget across algorithmic components.

    Parameters
    ----------
    error_init : float
        Error budget for initial state preparation.
    error_ham : float, optional
        Error budget for Hamiltonian time evolution. If omitted, this is inferred as
        ``error_yield - error_init`` for backward compatibility with older calls.
    error_qft_data : float, optional
        Error budget for the QFT on the data register. 
    error_qft_qpe : float
        Error budget for the QFT on the QPE register.
    error_d : float
        Error budget for quantum signal processing degree.
    error_yield : float
        Total allowed quantum-yield error.
    sim_time : float
        Simulation time in atomic units.
    lambda_ham : float
        Hamiltonian 1-norm.
    weight_theta, weight_T, weight_V : float
        Relative weights used to split the remaining Hamiltonian error budget between
        theta rotations, kinetic preparation, and potential preparation.
    error_rot, error_gamma, error_phi : float
        Error budgets for the rotation, gamma, and phi lookup tables, respectively.

    Returns
    -------
    tuple[float, float, float, float, float, float, float]
        ``(error_init, error_ham, error_qpe, error_qsp, error_theta, error_T, error_V)``.
    """
    if error_yield <= error_qft_qpe:
        print("error_yield:", error_yield)
        print("error_qft_qpe:", error_qft_qpe)
        raise ValueError(
            "error_yield must be larger than error_qft_qpe, otherwise no finite "
            "QPE register size can satisfy the error budget."
        )
    if error_ham < error_qsp:
        print("error_ham:", error_ham)
        print("error_qsp:", error_qsp)
        raise ValueError("error_ham must be at least error_qsp.")


    weight_sum = weight_theta + weight_T + weight_V

    qpe_discretization_budget = error_yield - error_qft_qpe - error_qsp - 2*(error_init + error_ham + error_qft_data)
    qpe_register_qubits = max(0, math.ceil(-math.log2(qpe_discretization_budget)))
    error_qpe = 1.0 / 2.0 ** (qpe_register_qubits) 

    remaining_ham_budget = error_ham - error_qsp
    error_theta = (weight_theta / weight_sum) * remaining_ham_budget / sim_time
    error_T = (weight_T / weight_sum) * remaining_ham_budget / sim_time
    error_V = (weight_V / weight_sum) * remaining_ham_budget / sim_time
    error_d = error_qsp - (error_rot + error_gamma + error_phi) * sim_time * lambda_ham

    
    if  2*(error_init + error_ham + error_qft_data) + error_qpe + error_qft_qpe > error_yield:
        raise ValueError("Errors exceed budget.")

    return error_init, error_ham, error_qpe, error_d, error_theta, error_T, error_V, error_rot


## 2. Mathematical helper functions

These utilities provide reusable formulas used by the time-evolution and quantum-yield resource estimates.


### QFT Toffoli count

$$C_{\mathrm{Toff}}(\mathrm{QFT}) = 4 n \left(\log_2(n / \epsilon) - 2\right) + 0.6 \log_2\left(\frac{n}{\epsilon}\log_2\left(\frac{n}{\epsilon}\right)\right).$$


In [3]:
def qft_toffoli_count(num_qubits, error_qft):
    """Estimate the Toffoli count for a quantum Fourier transform.

    Parameters
    ----------
    num_qubits : int
        Number of qubits in the QFT input register.
    error_qft : float
        Target QFT error tolerance.

    Returns
    -------
    float
        Estimated QFT Toffoli count.
    """
    term_1 = 4 * num_qubits * (math.log2(num_qubits / error_qft) - 2)
    term_2 = 0.6 * math.log2(num_qubits * math.log2(num_qubits / error_qft) / error_qft)
    return np.ceil(term_1 + term_2)



### Momentum-state success probability

$$P_{
u} \approx \frac{3}{8}\left[\mathrm{Ti}_2(3-\sqrt{8}) - G + \frac{\pi}{2}\log(1+\sqrt{2})\right].$$


In [4]:
def ti2(x):
    """
    Evaluate the inverse tangent integral Ti₂(x) = ∫₀ˣ arctan(t) / t dt.

    Parameters
    ----------
        x (float): upper integration limit

    Returns
    -------
        float: numerical approximation of Ti₂(x)
    """

    integrand = lambda t: np.arctan(t) / t if t != 0 else 1.0
    ti2_value, error = quad(integrand, 0, x, limit=100)
    return ti2_value

def calc_p_nu():
    """
    Approximate the momentum-state success probability constant Pₙ.

    Returns
    -------
        float: approximation of p_nu from the analytic expression in arXiv:1807.09802
    """

    sqrt2 = np.sqrt(2)
    sqrt8 = np.sqrt(8)
    G = ti2(1)
    term1 = ti2(3 - sqrt8)
    term2 = (np.pi / 2) * np.log(1 + sqrt2)
    p_nu = (3 / 8) * (term1 - G + term2)
    return p_nu


### Momentum-state ratio

$$r_{
u}=4\,\frac{7\cdot 2^{n_p+1}-9n_p-11-3\cdot 2^{-n_p}}{\lambda_{
u}}.$$


In [5]:
def ratio_nu(num_q, lambda_nu):
    """
    Compute the momentum state ratio used in normalization for block-encoding.

    Parameters
    ----------
        num_q (int): qubits per particle per dimension
        lambda_nu (float): momentum block-encoding parameter

    Returns
    -------
        float: ratio r_nu used for momentum-state preparation
    """

    r_nu = 4 * (7 * 2**(num_q + 1) - 9 * num_q - 11 - 3 * 2**(-num_q)) / lambda_nu
    return r_nu


### Register-size helper functions

- $n_M$: qubit count for momentum-state generation.
- $n_\theta$: qubit count for rotating between kinetic and potential terms.
- `num_time_steps`: number of QSP/qubitization walk queries.
- `Er`: QROM unpreparation helper.


In [6]:
def num_mom_qubits(lambda_v, error_V, num_q, lambda_nu):
    """
    Estimate the number of qubits required for momentum-state preparation.

    Parameters
    ----------
        lambda_v (float): potential block-encoding parameter
        error_V (float): allowed error for PREP_V in time evolution
        num_q (int): qubits per particle per dimension
        lambda_nu (float): momentum block-encoding parameter

    Returns
    -------
        int: number of qubits required for momentum-state generation
    """

    r_nu = ratio_nu(num_q, lambda_nu)
    n_M = np.ceil(math.log2(lambda_v * (r_nu / error_V)))
    return n_M

def num_rot_qubits(lambda_H, error_theta):
    """
    Estimate the number of qubits required for theta rotation in SELECT.

    Parameters
    ----------
        lambda_H (float): Hamiltonian block-encoding parameter
        error_theta (float): error tolerance for theta rotation

    Returns
    -------
        int: number of rotation qubits n_theta
    """

    n_theta = np.ceil(math.log2(lambda_H / error_theta))
    return n_theta

def num_time_steps(lambda_ham, sim_time, tol):
    
    """
    Compute number of time steps needed to simulate chemical dynamics for a molecule using qubitization

    Parameters
    ----------
        lambda_ham (float) - molecular hamiltonian normalization constant
        sim_time (float) - simulation time of chemical dynamics in au.
        tol (float) - error tolerance for time evolution

    Returns
    -------
        num_steps (int) - number of steps (calls to PREP-SEL-PREP blocks) needed
    """
    
    term_1 = lambda_ham * sim_time
    term_2 = np.log2(1/tol) / np.log2( np.log2(1/tol) )
    num_steps = np.ceil(term_1 + term_2)
 
    return num_steps

def Er(x):
    """
    Compute Er(x) = min_{k ∈ ℕ} (2^k + ⌈2^{-k} x⌉).

    Parameters
    ----------
        x (float): input parameter

    Returns
    -------
        float: minimum value of the expression across natural k
    """

    def calculate_term(k, x):
        power_term = 2 ** k
        ceiling_term = math.ceil((2 ** -k) * x)
        return power_term + ceiling_term

    min_value = calculate_term(0, x)
    k = 1

    while k < 100:
        current_value = calculate_term(k, x)
        if current_value < min_value:
            min_value = current_value
        elif current_value > min_value:
            break
        k += 1

    return min_value


## 3. Time-evolution resource estimates

Estimate Toffoli counts and logical-qubit counts for the qubitization walk operator and QSP time evolution.


### Toffoli-count primitives and propagator cost


In [7]:
def toff_count_uniform_state(n, b_r):
    """
    Estimate Toffoli cost for preparing a uniform superposition state.

    Parameters
    ----------
        n (int): number of non-zero coefficients
        b_r (int): number of rotation bits used in state preparation

    Returns
    -------
        float: Toffoli cost for uniform state preparation
    """

    toff_count = np.ceil(3 * int.bit_length(n) + 2 * b_r - 9)
    return toff_count

def toff_count_comparator(n):
    """
    Estimate Toffoli cost for an n-qubit inequality comparator.

    Parameters
    ----------
        n (int): number of qubits in the comparator

    Returns
    -------
        float: Toffoli cost for comparator operation
    """

    toff_count = n - 1
    return toff_count

def toff_count_qrom(n):
    """
    Estimate Toffoli cost for QROM address iteration.

    Parameters
    ----------
        n (int): number of coefficients in QROM

    Returns
    -------
        float: Toffoli cost for QROM iteration
    """

    toff_count = n - 1
    return toff_count

def toff_count_unprep_qrom(n):
    """
    Estimate Toffoli cost for erasing a QROM register.

    Parameters
    ----------
        n (int): number of coefficients in QROM

    Returns
    -------
        float: Toffoli cost to unprepare QROM data
    """

    toff_count = np.ceil(Er(n))
    return toff_count



def time_evol_gates(num_q, num_elec, num_particles, n_theta, n_M, a, query_cost, lambda_t, error_T, b_R, error_rot):
    """
    Estimate Toffoli gates required for one full time evolution instance.

    Parameters
    ----------
        num_q (int): number of qubits per particle per dimension
        num_elec (int): number of electrons
        num_particles (int): total particles (electrons + nuclei)
        n_theta (int): rotation qubits between T and V operators
        n_M (int): momentum-state preparation qubits
        a (int): amplitude amplification parameter
        query_cost (int): number of walk operator query steps
        lambda_t (float): kinetic block-encoding parameter
        error_T (float): allowed kinetic PREP error
        b_R (int): rotation precision bits for uniform state preparation
        error_rot (float): rotation error tolerance for QSP

    Returns
    -------
        tuple: (Toffoli count for full propagator, Toffoli count for controlled qubiterate)
    """

    eta = num_particles
    n_eta = np.ceil(math.log2(num_particles))
    epsilon_t = error_T
    mu_T = np.ceil(math.log2(lambda_t / epsilon_t))

    # State preparation gate counts for PREP_T and PREP_V
    Toff_count_theta_state = n_theta - 3
    Toff_count_w_state = 13
    Toff_count_rs_state = np.ceil(2 * (num_q - 2))
    Toff_count_mass_state = np.ceil(eta + mu_T + 4 * n_eta + 7)
    Toff_count_momentum_state = np.ceil(a * (3 * num_q**2 + 11 * num_q + 2 + 4 * n_M * (num_q + 1)))
    Toff_count_zeta_state = toff_count_uniform_state(2 * num_elec, b_R) + toff_count_qrom(2 * num_elec)
    Toff_count_comparator_zeta = toff_count_comparator(n_eta)
    Toff_count_coulomb_state = Toff_count_comparator_zeta + 2 * Toff_count_zeta_state

    Toff_count_PREP_T = Toff_count_w_state + Toff_count_rs_state + Toff_count_mass_state
    Toff_count_PREP_V = Toff_count_momentum_state + Toff_count_coulomb_state
    Toff_count_PREP_H = Toff_count_theta_state + Toff_count_PREP_T + Toff_count_PREP_V

    # Unpreparation gate counts
    Toff_count_unprep_theta_state = n_theta - 3
    Toff_count_unprep_w_state = 13
    Toff_count_unprep_rs_state = np.ceil(2 * (num_q - 2))
    Toff_count_unprep_mass_state = np.ceil(Er(eta) + 4 * n_eta + 9)
    Toff_count_unprep_momentum_state = np.ceil(a * (4 * (num_q - 1)))
    Toff_count_unprep_zeta_state = toff_count_uniform_state(2 * num_elec, b_R) + toff_count_unprep_qrom(2 * num_elec)
    Toff_count_unprep_coulomb_state = 2 * Toff_count_unprep_zeta_state

    Toff_count_UNPREP_T = Toff_count_unprep_w_state + Toff_count_unprep_rs_state + Toff_count_unprep_mass_state
    Toff_count_UNPREP_V = Toff_count_unprep_momentum_state + Toff_count_unprep_coulomb_state
    Toff_count_UNPREP_H = Toff_count_unprep_theta_state + Toff_count_UNPREP_T + Toff_count_UNPREP_V

    # SEL and controlled SEL costs
    Toff_count_SWAPS = 3 * eta * num_q
    Toff_count_UnaryIt = eta - 2
    Toff_count_ctrl_SEL_T = 5 * (num_q - 1) + 2
    Toff_count_ctrl_SEL_V = 24 * num_q
    Toff_count_SEL_H = 6 * (Toff_count_SWAPS + Toff_count_UnaryIt) + Toff_count_ctrl_SEL_T + Toff_count_ctrl_SEL_V
    Toff_count_ctrl_SEL_H = Toff_count_SEL_H + 1

    # Reflection and controlled qubiterate costs
    Toff_count_ctrl_R_0 = n_eta + 6 * num_q + n_M + 2 * (2 * num_elec).bit_length() + 10
    Toff_count_ctrl_qubiterate = (
        Toff_count_ctrl_SEL_H + Toff_count_PREP_H + Toff_count_UNPREP_H + Toff_count_ctrl_R_0
    )

    # QSP rotation costs
    Toff_count_QSP_rotations = np.ceil(0.5 * (0.56 * np.log2(1 / error_rot) + 5.3))

    # Total propagator cost per time step
    Toff_count_U_prop = (
        2 * Toff_count_ctrl_qubiterate
        + query_cost * (Toff_count_ctrl_qubiterate - 1)
        + (query_cost + 1) * Toff_count_QSP_rotations
    )

    return Toff_count_U_prop, Toff_count_ctrl_qubiterate


### Logical-qubit count for time evolution

$$C_{\mathrm{anc}}(U_{\mathrm{prop}})=1+\max\{C_{\mathrm{anc}}(\mathrm{PREP}_H)+C_{\mathrm{anc}}(\mathrm{SEL}_H), C_{\mathrm{anc}}(\mathcal{R}_0^{(\mathcal{W})})\}.$$


In [8]:
def time_evol_qubits(num_q, num_particles, num_electrons, n_theta, n_M, lambda_t, error_T, error_rot):
    """
    Estimate logical qubits required for the qubitization walk operator.

    Parameters
    ----------
        num_q (int): qubits per particle per dimension
        num_particles (int): total number of particles
        num_electrons (int): number of electrons
        n_theta (int): theta rotation qubits
        n_M (int): momentum state preparation qubits
        lambda_t (float): kinetic block-encoding parameter
        error_T (float): kinetic PREP error tolerance
        error_rot (float): signal processing rotation error tolerance

    Returns
    -------
        tuple: (total qubits, ancilla qubits to unprepare state prep)
    """

    n_eta = (num_particles - 1).bit_length()
    n_grad = max(n_theta, math.log2(1 / error_rot))
    epsilon_t = error_T
    mu_T = np.ceil(math.log2(lambda_t / epsilon_t))

    Qubits_prep = np.ceil(
        3 * (num_q**2)
        + 10 * num_q
        + 5 * n_M
        + 4 * n_M * num_q
        + 6 * math.log2(2 * num_electrons)
        + 3 * n_eta
        + 4
        + 2 * num_q
        + 4 * n_eta
        + 3 * mu_T
        + 19
        + n_grad
    )

    Qubits_sel = 5 * num_q + n_eta + 11
    prep_dagger_anc = np.ceil(
        n_eta
        + mu_T
        + 6 * num_q
        + n_M
        + 2 * (2 * num_electrons).bit_length()
        + 9
    )

    Qubits_R0 = np.ceil(2 * (
        n_eta
        + mu_T
        + 6 * num_q
        + n_M
        + 2 * (2 * num_electrons).bit_length()
        + 10
    ))

    return 1 + max(Qubits_prep + Qubits_sel, Qubits_R0), prep_dagger_anc


### Amplitude-amplification optimization

Compare the supported amplitude-amplification choices and keep the lower-cost time-evolution estimate.


In [9]:
def compute_resources_for_a(sim_params, psi4_file, isp_resource_file, 
                            num_nuc, error_theta, error_T, error_V, 
                            error_d, error_rot, a):
    """
    Compute Toffoli gate estimates for a given amplitude amplification parameter.

    Parameters
    ----------
        sim_params (dict): simulation parameters dictionary
        psi4_file (str): path to the psi4 input file
        isp_resource_file (str): path to the ISP resource file
        num_nuc (int): number of nuclei
        error_theta (float): allowed theta rotation error
        error_T (float): allowed kinetic PREP error
        error_V (float): allowed potential PREP error
        error_d (float): allowed QSP degree error
        a (int): amplitude amplification parameter
        error_rot (float): rotation error


    Returns
    -------
        tuple: (total_gates, gates_per_call, lambda_molec, n_theta, n_M, lambda_t)
            total_gates (float): total Toffoli gates for the simulation
            gates_per_call (float): Toffoli gates per walk operator call
            lambda_molec (float): molecular Hamiltonian block-encoding parameter
            n_theta (int): rotation qubits for theta control
            n_M (int): momentum preparation qubits
            lambda_t (float): kinetic block-encoding parameter
    """

    _, molec_masses, molec_labels = parse_psi4.get_molec_data(psi4_file, num_nuc)
    num_q = int(parse_isp.parse_isp_file(isp_resource_file)["n_p"])
    num_elec = sim_params["electrons"]
    sim_time = physical_consts.femto2au(sim_params["time"])
    qsp_tol = error_d
    L = parse_isp.parse_isp_file(isp_resource_file)["L"]
    b_R = sim_params["b_R"]

    nuc_charges = charge_ratio.compute_charge_list(sim_params, molec_labels)
    p_zeta = charge_ratio.compute_charge_ratio(sim_params, molec_labels)
    lambda_nu = lam_nu.get_lambda_nu()[num_q - 1]

    num_particles = num_elec + num_nuc
    vol = L**3

    lambda_t = lam.lambda_T(num_elec, num_q, vol, molec_masses)
    lambda_v = lam.lambda_V(num_elec, lambda_nu, vol, nuc_charges)
    n_M = num_mom_qubits(lambda_v, error_V, num_q, lambda_nu)

    p_nu = calc_p_nu()

    lambda_1 = lam.lambda_ham(
        vol, num_q, num_elec, num_nuc, molec_masses, lambda_nu, nuc_charges, b_R
    )
    lambda_2 = lam.lambda_alt(
        vol, b_R, num_nuc, num_elec, lambda_nu, nuc_charges, p_nu, a, p_zeta
    )
    lambda_molec = max(lambda_1, lambda_2)

    time_steps = num_time_steps(lambda_molec, sim_time, qsp_tol)
    n_theta = num_rot_qubits(lambda_molec, error_theta)

    total_gates, gates_per_call = time_evol_gates(
        num_q,
        num_elec,
        num_particles,
        n_theta,
        n_M,
        a,
        time_steps,
        lambda_t,
        error_T,
        b_R,
        error_rot,
    )

    return total_gates, gates_per_call, lambda_molec, n_theta, n_M, lambda_t

def time_evolution_optimization(sim_params, psi4_file, isp_resource_file, num_nuc, error_theta, 
                                error_T, error_V, error_d, error_rot):
    """
    Optimize time-evolution resources by selecting the best amplitude amplification factor.

    Parameters
    ----------
        sim_params (dict): simulation parameters dictionary
        psi4_file (str): psi4 input file path
        isp_resource_file (str): ISP resource file path
        num_nuc (int): number of nuclei in the molecule
        error_theta (float): theta rotation error budget
        error_T (float): kinetic PREP error budget
        error_V (float): potential PREP error budget
        error_d (float): QSP degree error budget
        error_rot (float): rotation error budget

    Returns
    -------
        tuple: (optimal_total_gates, n_theta, n_M, a, lambda_molec, lambda_t)
    """

    total_gates_a1, _, lambda_molec_1, n_theta_1, n_M_1, lambda_t = compute_resources_for_a(
        sim_params,
        psi4_file,
        isp_resource_file,
        num_nuc,
        error_theta,
        error_T,
        error_V,
        error_d,
        error_rot,
        a=1
    )

    total_gates_a3, _, lambda_molec_3, n_theta_3, n_M_3, lambda_t = compute_resources_for_a(
        sim_params,
        psi4_file,
        isp_resource_file,
        num_nuc,
        error_theta,
        error_T,
        error_V,
        error_d,
        error_rot,
        a=3,
    )

    if total_gates_a1 <= total_gates_a3:
        return total_gates_a1, n_theta_1, n_M_1, 1, lambda_molec_1, lambda_t
    else:
        return total_gates_a3, n_theta_3, n_M_3, 3, lambda_molec_3, lambda_t


## 4. Quantum-yield resource estimates

Combine initial-state-preparation, time-evolution, QPE/QFT, and bond-indicator costs into the quantum-yield estimate.


In [11]:
def get_quantum_yield(num_q, num_particles, num_nuclei, n_p_isp, u_ham_toffoli_count, 
                      u_init_toffoli_count, num_bonds, error_qpe, error_qft_data, error_qft_qpe):
    """
    Compute quantum yield resource estimates for a molecular bond.

    Parameters
    ----------
        num_q (int): number of qubits per particle per spatial dimension
        num_particles (int): total number of particles in the molecule
        num_nuclei (int): number of nuclei in the molecule
        n_p_isp (int): number of ISP qubits per nuclear register
        u_ham_toffoli_count (int): Toffoli gates for time-evolution Hamiltonian
        u_init_toffoli_count (int): Toffoli gates for initial state preparation
        num_bonds (int): number of bond distances being evaluated
        error_qpe (float): quantum phase estimation error budget
        error_qft_data (float): QFT error budget for data registers
        error_qft_qpe (float): QFT error budget for the QPE register

    Returns
    -------
        quantum_yield_toffoli_count (float): total Toffoli gates for yield computation
        yield_ancilla_qubits (int): ancilla qubit count for yield computation
        u_pi_qubit_count (int): qubits required for the U_pi operator
    """

    # QPE register size and number of controlled walk operator calls
    qpe_ancilla_qubits = np.ceil(math.log2(1 / error_qpe))
    u_walk_calls = np.power(2, qpe_ancilla_qubits - 1)
    print("u_walk_calls:", u_walk_calls)

    num_elec = num_particles - num_nuclei
    num_nuclei_involved = 2  # two nuclei are involved in each bond distance evaluation

    qubits_per_particle = 3 * num_q
    nuclei_data_reg_size = num_nuclei * qubits_per_particle

    flag_reg_size = 1  # single flag qubit for QAE

    walk_qft_toffoli_count = qft_toffoli_count(nuclei_data_reg_size, error_qft_data)
    qpe_qft_toffoli_count = qft_toffoli_count(qpe_ancilla_qubits, error_qft_qpe)

    indicator_toffoli_count = int(
        3 * num_bonds * (2 * num_q**2 + 2 * num_q - 3) + (3 * num_nuclei_involved) * (num_q - 2))

    u_pi_qubit_count = 3 * (num_q**2) - num_q
    ctrl_r0_qae_toffoli_count = 3 * (num_elec * num_q + num_nuclei * n_p_isp)

    u_walk_toffoli_count = (
        2 * (u_init_toffoli_count + u_ham_toffoli_count + indicator_toffoli_count + walk_qft_toffoli_count)
        + ctrl_r0_qae_toffoli_count
    )

    quantum_yield_toffoli_count = (
        u_walk_calls * u_walk_toffoli_count
        + (u_init_toffoli_count + u_ham_toffoli_count + qpe_qft_toffoli_count + walk_qft_toffoli_count)
    )

    yield_ancilla_qubits = flag_reg_size + qpe_ancilla_qubits

    return quantum_yield_toffoli_count, yield_ancilla_qubits, u_pi_qubit_count


## 5. Top-level Function

Use `compute_quantum_yield_resources(...)` to return a structured dictionary, or `quantum_yield_resources(...)` to write a formatted text report.


In [14]:
def compute_quantum_yield_resources(parameters_file, isp_resource_file, psi4_file, error_budget_file):
    """
    Compute all resource estimates needed to calculate the quantum yield.

    Parameters
    ----------
        parameters_file (str): path to the simulation parameters file
        isp_resource_file (str): path to the ISP resource file
        psi4_file (str): path to the psi4 output file

    Returns
    -------
        dict: resource dictionary containing qubits, ancilla, gate counts, and error summaries
    """
    
    # Read in simulation and ISP data files
    sim_parameters = io.read_file(parameters_file)
    isp_resources = io.read_file(isp_resource_file)
    error_budget = io.read_file(error_budget_file)
    
    # Unpack particle and grid parameters
    num_elec = sim_parameters["electrons"]
    num_nuc = sim_parameters["nuclei"]
    num_bonds = sim_parameters["num_bonds"]
    qubits_per_reg = isp_resources["n_p"]
    n_p_isp = isp_resources["n_p_isp"]
    sim_time = physical_consts.femto2au(sim_parameters["time"])
    L = isp_resources["L"]
    vol = L**3
    _, molec_masses, molec_labels = parse_psi4.get_molec_data(psi4_file, num_nuc)
    lambda_nu = lam_nu.get_lambda_nu()[qubits_per_reg - 1]
    charges = charge_ratio.compute_charge_list(sim_parameters, molec_labels)
    b_R = sim_parameters["b_R"]
    

    # Get error budgets for the full simulation stack
    error_qft_data = error_budget["basis_trans_budget"]
    error_qft_qpe = error_budget["qae_qft_budget"]
    error_yield = error_budget["total_budget"]
    error_qsp = error_budget["qsp_budget"]

    error_init = error_budget.get("isp_budget")
    error_ham = error_budget.get("ham_budget")

    lambda_ham = lam.lambda_ham(vol, qubits_per_reg, num_elec, num_nuc, molec_masses, lambda_nu, charges, b_R)
    
    error_init, error_ham, error_qpe, error_d, error_theta, error_T, error_V, error_rot = get_error_budget(
            error_init, error_ham, error_qft_data, error_qft_qpe, error_qsp, error_yield, sim_time, lambda_ham
        )
    
    # ISP resources from electronic and nuclear precomputation
    elec_isp_ancilla = isp_resources["elec_isp_anc"]
    elec_isp_toffoli_count = isp_resources["elec_isp_Toff"]
    nuc_isp_ancilla = isp_resources["nuc_isp_anc"]
    nuc_isp_toffoli_count = isp_resources["nuc_isp_Toff"]

    exterior_grid_ancilla = 3 * num_nuc * (n_p_isp - qubits_per_reg)
    
    # Basic qubit counts for the full system
    num_particles = num_elec + num_nuc
    state_vec_qubits = 3 * num_particles * qubits_per_reg
    elec_qubits = 3 * num_elec * qubits_per_reg
    nuc_qubits = 3 * num_nuc * qubits_per_reg
    
    # Total ISP resource counts
    isp_toffoli_count = elec_isp_toffoli_count + nuc_isp_toffoli_count
    isp_ancilla = max(elec_isp_ancilla , nuc_isp_ancilla)
    
    # Time evolution resource estimates
    time_evolution_toffoli_count, n_theta, n_M, a, lambda_molec, lambda_t = time_evolution_optimization(
        sim_parameters, psi4_file, isp_resource_file, num_nuc, error_theta, error_T, error_V, error_d, error_rot
    )
    tevol_qubits, prep_dagger_anc = time_evol_qubits(
        qubits_per_reg, num_particles, num_elec, n_theta, n_M, lambda_t, error_T, error_rot
    )
    tevol_ancilla = tevol_qubits
    
    # Quantum yield specific resources
    quantum_yield_toffoli_count, quantum_yield_ancilla_qubitsilla, u_pi_qubit_count = get_quantum_yield(
        qubits_per_reg, num_particles, num_nuc, n_p_isp, time_evolution_toffoli_count,
        isp_toffoli_count, num_bonds, error_qpe, error_qft_data, error_qft_qpe
    )
    
    # Combine ancilla and qubit requirements into totals
    total_ancilla = np.ceil(
        max(isp_ancilla - exterior_grid_ancilla, tevol_ancilla, u_pi_qubit_count)
        + quantum_yield_ancilla_qubitsilla + exterior_grid_ancilla
    )
    total_qubits = total_ancilla + state_vec_qubits
    
    resource_dictionary = {
        "total_resources": {
            "Total Qubits": int(total_qubits),
            "State Vector Qubits": int(state_vec_qubits),
            "Electron Qubits": int(elec_qubits),
            "Nuclear Qubits": int(nuc_qubits),
            "Ancilla Qubits": int(total_ancilla),
            "Toffoli Count": int(quantum_yield_toffoli_count),
            "Lambda": lambda_molec,
            "delta_L": L / (2**qubits_per_reg)
        },
        "electronic_resources": {
            "Ancilla Qubits": int(elec_isp_ancilla),
            "Toffoli Count": int(elec_isp_toffoli_count)
        },
        "nuclear_resources": {
            "Ancilla Qubits": int(nuc_isp_ancilla),
            "Toffoli Count": int(nuc_isp_toffoli_count)
        },
        "time_evolution_resources": {
            "Qubits per dimension": int(qubits_per_reg),
            "Simulation cell length": L,
            "Ancilla Qubits": int(tevol_ancilla),
            "Toffoli Count": int(time_evolution_toffoli_count)
        },
        "error_budget": {
            "Initial State Preparation Error": error_init,
            "Hamiltonian Simulation Error": error_ham,
            "QPE Error": error_qpe,
            "Total Quantum Yield Error": error_yield
        },
        "simulation_parameters": sim_parameters
    }
    
    return resource_dictionary


In [15]:
def quantum_yield_resources(
    parameters_file,
    isp_resource_file,
    psi4_file,
    error_budget_file,
    output_file,
    molecule_name,
):
    """Compute and write quantum-yield resource estimates.

    The output report includes the calculation date and time, the final
    resource estimates, and a copy of each input file used to configure
    the calculation.

    Parameters
    ----------
    parameters_file : str or pathlib.Path
        Path to the simulation-parameters input file.
    isp_resource_file : str or pathlib.Path
        Path to the ISP resource file.
    psi4_file : str or pathlib.Path
        Path to the Psi4 input file.
    error_budget_file : str or pathlib.Path
        Path to the error-budget input file.
    output_file : str or pathlib.Path
        Path to the output text file.
    molecule_name : str
        Molecule name displayed in the output-file header.

    Returns
    -------
    int
        Returns 0 after successfully writing the resource-estimate file.
    """
    # Record when the resource estimate was generated. The local UTC offset
    # is included so that the timestamp remains unambiguous.
    calculation_time = datetime.now().astimezone()

    # Compute all estimated resources.
    resources = compute_quantum_yield_resources(
        parameters_file,
        isp_resource_file,
        psi4_file,
        error_budget_file,
    )

    # Write the formatted resource-estimate report.
    with open(output_file, "w") as file:
        file.write(f"{molecule_name} Quantum Yield Resource Estimate\n")
        file.write(
            "Generated: "
            f"{calculation_time.strftime('%Y-%m-%d %H:%M:%S %Z (UTC%z)')}\n"
        )

        file.write("\n------------\nTotal Resources\n")
        for key, value in resources["total_resources"].items():
            if isinstance(value, (int, float)) and abs(value) > 9999:
                file.write(f"{key}={value:.2e}\n")
            else:
                file.write(f"{key}={value}\n")

        file.write("\n------------\nElectronic ISP Breakdown\n")
        file.write("Electronic:\n")
        for key, value in resources["electronic_resources"].items():
            if isinstance(value, (int, float)) and abs(value) > 9999:
                file.write(f"{key}={value:.2e}\n")
            else:
                file.write(f"{key}={value}\n")

        file.write("\n------------\nNuclear ISP Breakdown\n")
        file.write("Nuclear:\n")
        for key, value in resources["nuclear_resources"].items():
            if isinstance(value, (int, float)) and abs(value) > 9999:
                file.write(f"{key}={value:.2e}\n")
            else:
                file.write(f"{key}={value}\n")

        file.write("\n------------\nTime Evolution Breakdown\n")
        for key, value in resources["time_evolution_resources"].items():
            if isinstance(value, (int, float)) and abs(value) > 9999:
                file.write(f"{key}={value:.2e}\n")
            else:
                file.write(f"{key}={value}\n")

        file.write("\n------------\nErrors\n")
        for key, value in resources["error_budget"].items():
            file.write(f"{key}={value}\n")

        # Append copies of the input files after the calculated results.
        file.write("\n============================\n")
        file.write("Input File Stamp\n")
        file.write("============================\n")

        input_files = (
            parameters_file,
            error_budget_file,
            psi4_file,
        )

        for input_file in input_files:
            input_path = Path(input_file)

            file.write(f"\n{input_path.name}\n")
            file.write(f"------------\n")
            with open(input_path, "r") as source_file:
                contents = source_file.read()

            file.write(contents)

            # Ensure that the next section begins on a new line.
            if contents and not contents.endswith("\n"):
                file.write("\n")

    return 0